# Configuração e Carga de Dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import json
import os

# 1. Carregar Config
with open('config/settings.json', 'r') as f:
    config = json.load(f)

video_id = config['video_id']
csv_path = f'data/processed/chat_{video_id}_processed.csv'

df = pd.read_csv(csv_path)

def t2s(t): 
    parts = list(map(int, t.split(':')))
    return parts[0]*3600 + parts[1]*60 + parts[2] if len(parts)==3 else parts[0]*60 + parts[1]

v_start = t2s(config['game_start_time'])
fh_end = (t2s(config['first_half_end']) - v_start) / 60
sh_start = (t2s(config['second_half_start']) - v_start) / 60
sh_end = (t2s(config['second_half_end']) - v_start) / 60

df['min_lin'] = (df['timestamp_jogo_segundos'] // 60).astype(int)
vol = df.groupby('min_lin').size().reindex(range(int(df['min_lin'].min()), int(df['min_lin'].max()) + 1), fill_value=0)

## Gráfico 1: Picos Instantâneos (Volume Bruto)

In [ ]:
diff_raw = vol.diff()
spikes_raw = vol[diff_raw >= 15]

plt.figure(figsize=(15, 6))
vol.plot(color='black', alpha=0.3, label='Bruto')
plt.axvspan(0, fh_end, color='lightgreen', alpha=0.2, label='1º Tempo')
plt.axvspan(sh_start, sh_end, color='lightblue', alpha=0.2, label='2º Tempo')

plt.scatter(spikes_raw.index, spikes_raw.values, color='red', s=40, zorder=5, label='Aumentos Brutos')
for m, v in spikes_raw.items():
    label = df[df['min_lin'] == m]['label_partida'].iloc[0] if m in df['min_lin'].values else f"{int(m)}'"
    plt.annotate(label, (m, v), textcoords="offset points", xytext=(0,10), ha='center', color='red', fontsize=9, fontweight='bold')

plt.title(f'Picos de Volume Instantâneos - {video_id}', fontsize=14)
plt.legend()
plt.show()

## Gráfico 2: Aumentos Súbitos na Tendência

In [ ]:
window_size = 5
vol_smoothed = vol.rolling(window=window_size, center=True).mean()
diff_smooth = vol_smoothed.diff()
threshold_smooth = 5
spikes_smooth_trend = vol_smoothed[diff_smooth >= threshold_smooth]

plt.figure(figsize=(15, 6))
plt.plot(vol_smoothed.index, vol_smoothed.values, color='darkblue', linewidth=2.5, label='Tendência Suavizada')
plt.axvspan(0, fh_end, color='lightgreen', alpha=0.2, label='1º Tempo')
plt.axvspan(sh_start, sh_end, color='lightblue', alpha=0.2, label='2º Tempo')

plt.scatter(spikes_smooth_trend.index, spikes_smooth_trend.values, color='magenta', s=50, edgecolors='black', zorder=5, label='Início de Hype (Tendência)')

for m, v in spikes_smooth_trend.items():
    label = df[df['min_lin'] == m]['label_partida'].iloc[0] if m in df['min_lin'].values else f"{int(m)}'"
    plt.annotate(label, (m, v), textcoords="offset points", xytext=(0,10), ha='center', color='magenta', fontsize=9, fontweight='bold')

plt.title(f'Identificação de Início de Hype (Tendência) - {video_id}', fontsize=14)
plt.legend()
plt.show()